In [1]:
%%writefile assignment_jax.py
"""
ELG5214 Assignment 1
- Jax implementation with adam
- Pure JAX MLP
- MNIST loading + preprocessing
- Batch sizes: 64, 256, 1024
- Timing: first epoch + steady-state epoch time
- Final test accuracy
- Saves CSV + plots
"""

from __future__ import annotations
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp


# Config
SEED = 42
EPOCHS = 50
LR = 1e-3
H1 = 256
H2 = 128
NUM_CLASSES = 10
BATCH_SIZES = [64, 256, 1024]
VAL_SIZE = 10_000

OUT_DIR = "outputs_jax"


# Data: load MNIST and preprocess
def one_hot(y: np.ndarray, k: int = 10) -> np.ndarray:
    out = np.zeros((y.shape[0], k), dtype=np.float32)
    out[np.arange(y.shape[0]), y] = 1.0
    return out

def load_mnist(seed: int = 42, val_size: int = 10_000):
    # Keras MNIST loader works in Colab easily
    from tensorflow.keras.datasets import mnist
    (x_train, y_train), (x_test, y_test) = mnist.load_data()

    x_train = (x_train.astype(np.float32) / 255.0).reshape(-1, 784)
    x_test  = (x_test.astype(np.float32) / 255.0).reshape(-1, 784)

    rng = np.random.default_rng(seed)
    idx = np.arange(x_train.shape[0])
    rng.shuffle(idx)
    x_train, y_train = x_train[idx], y_train[idx]

    x_val, y_val = x_train[:val_size], y_train[:val_size]
    x_train, y_train = x_train[val_size:], y_train[val_size:]

    return (x_train, one_hot(y_train)), (x_val, one_hot(y_val)), (x_test, one_hot(y_test))


# JAX MLP from scratch
def init_params(key: jax.random.PRNGKey, in_dim: int, h1: int, h2: int, out_dim: int):
    # He init for ReLU layers
    k1, k2, k3 = jax.random.split(key, 3)
    w1 = jax.random.normal(k1, (in_dim, h1)) * jnp.sqrt(2.0 / in_dim)
    b1 = jnp.zeros((h1,))
    w2 = jax.random.normal(k2, (h1, h2)) * jnp.sqrt(2.0 / h1)
    b2 = jnp.zeros((h2,))
    w3 = jax.random.normal(k3, (h2, out_dim)) * jnp.sqrt(2.0 / h2)
    b3 = jnp.zeros((out_dim,))
    return {"w1": w1, "b1": b1, "w2": w2, "b2": b2, "w3": w3, "b3": b3}

def forward(params, x):
    z1 = x @ params["w1"] + params["b1"]
    a1 = jax.nn.relu(z1)
    z2 = a1 @ params["w2"] + params["b2"]
    a2 = jax.nn.relu(z2)
    logits = a2 @ params["w3"] + params["b3"]
    return logits

def loss_fn(params, x, y_onehot):
    logits = forward(params, x)
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    return -jnp.mean(jnp.sum(y_onehot * log_probs, axis=-1))

def acc_fn(params, x, y_onehot):
    logits = forward(params, x)
    pred = jnp.argmax(logits, axis=-1)
    true = jnp.argmax(y_onehot, axis=-1)
    return jnp.mean(pred == true)

@jax.jit
def train_step(params, x, y, lr: float):
    grads = jax.grad(loss_fn)(params, x, y)
    params = {k: params[k] - lr * grads[k] for k in params}
    return params


#adam
def adam_init(params):
    m = {k: jnp.zeros_like(v) for k, v in params.items()}
    v = {k: jnp.zeros_like(v) for k, v in params.items()}
    t = jnp.array(0, dtype=jnp.int32)
    return (m, v, t)

@jax.jit
def adam_step(params, opt_state, x, y, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    m, v, t = opt_state
    t = t + 1
    grads = jax.grad(loss_fn)(params, x, y)

    m = {k: beta1 * m[k] + (1.0 - beta1) * grads[k] for k in params}
    v = {k: beta2 * v[k] + (1.0 - beta2) * (grads[k] * grads[k]) for k in params}

    mhat = {k: m[k] / (1.0 - beta1 ** t) for k in params}
    vhat = {k: v[k] / (1.0 - beta2 ** t) for k in params}

    params = {k: params[k] - lr * mhat[k] / (jnp.sqrt(vhat[k]) + eps) for k in params}
    return params, (m, v, t)


# Training
def run_experiment(x_train, y_train, x_val, y_val, x_test, y_test, batch_size: int):
    key = jax.random.PRNGKey(SEED)
    params = init_params(key, 784, H1, H2, NUM_CLASSES)
    opt_state = adam_init(params)


    # Put val/test on device once
    x_val_d, y_val_d = jax.device_put(x_val), jax.device_put(y_val)
    x_test_d, y_test_d = jax.device_put(x_test), jax.device_put(y_test)

    train_losses = []
    val_accs = []
    epoch_times = []

    n = x_train.shape[0]
    for ep in range(EPOCHS):
        t0 = time.perf_counter()

        rng = np.random.default_rng(SEED + ep)
        idx = np.arange(n)
        rng.shuffle(idx)
        x_shuf = x_train[idx]
        y_shuf = y_train[idx]

        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            xb = jax.device_put(x_shuf[start:end])
            yb = jax.device_put(y_shuf[start:end])
            params, opt_state = adam_step(params, opt_state, xb, yb, LR)


        # sync so timing is accurate
        jax.block_until_ready(params["w1"])

        t1 = time.perf_counter()
        epoch_times.append(t1 - t0)

        # curves
        tr_loss = float(loss_fn(params,
                               jax.device_put(x_shuf[:2048]),
                               jax.device_put(y_shuf[:2048])))
        va = float(acc_fn(params, x_val_d, y_val_d))

        train_losses.append(tr_loss)
        val_accs.append(va)

    first_epoch = float(epoch_times[0])
    steady_state = float(np.mean(epoch_times[1:])) if EPOCHS > 1 else first_epoch
    test_acc = float(acc_fn(params, x_test_d, y_test_d))

    return {
        "first_epoch_time_s": first_epoch,
        "steady_state_epoch_time_s": steady_state,
        "final_test_acc": test_acc,
        "train_loss_curve": train_losses,
        "val_acc_curve": val_accs,
    }

# Main: run all batch sizes
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    (x_train, y_train), (x_val, y_val), (x_test, y_test) = load_mnist(SEED, VAL_SIZE)

    rows = []
    curves = {}

    for bs in BATCH_SIZES:
        res = run_experiment(x_train, y_train, x_val, y_val, x_test, y_test, bs)
        rows.append({
            "framework": "JAX",
            "batch_size": bs,
            "first_epoch_time_s": res["first_epoch_time_s"],
            "steady_state_epoch_time_s": res["steady_state_epoch_time_s"],
            "final_test_acc": res["final_test_acc"],
        })
        curves[str(bs)] = res

    df = pd.DataFrame(rows).sort_values(["batch_size"]).reset_index(drop=True)
    df.to_csv(os.path.join(OUT_DIR, "perf_results_jax.csv"), index=False)
    print("\n=== JAX Results ===")
    print(df)


    # Training Loss
    plt.figure()
    for bs in BATCH_SIZES:
        loss_curve = curves[str(bs)]["train_loss_curve"]
        plt.plot(loss_curve, label=f"bs={bs}")

    plt.xlabel("Epoch")
    plt.ylabel("Train Loss")
    plt.title("JAX Training Loss (All Batch Sizes)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "train_loss_all_batch_sizes.png"), dpi=150)
    plt.close()

    # Validation Accuracy
    plt.figure()
    for bs in BATCH_SIZES:
        acc_curve = curves[str(bs)]["val_acc_curve"]
        plt.plot(acc_curve, label=f"bs={bs}")

    plt.xlabel("Epoch")
    plt.ylabel("Validation Accuracy")
    plt.title("JAX Validation Accuracy (All Batch Sizes)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "val_acc_all_batch_sizes.png"), dpi=150)
    plt.close()

    print(f"\nSaved combined plots in: {OUT_DIR}/")


if __name__ == "__main__":
    main()


Writing assignment_jax.py


In [2]:
%%writefile assignment_torch.py
"""
ELG5214 Assignment 1
- PyTorch
- Same architecture as JAX: 784 -> 256 -> 128 -> 10 with ReLU
- Batch sizes: 64, 256, 1024
- Timing: first epoch + steady-state epoch time
- Final test accuracy
- Saves CSV + plots
"""

from __future__ import annotations
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F


# Config

SEED = 42
EPOCHS = 50
LR = 1e-3
H1 = 256
H2 = 128
NUM_CLASSES = 10
BATCH_SIZES = [64, 256, 1024]
VAL_SIZE = 10_000

OUT_DIR = "outputs_torch"



# Data
def one_hot(y: np.ndarray, k: int = 10) -> np.ndarray:
    out = np.zeros((y.shape[0], k), dtype=np.float32)
    out[np.arange(y.shape[0]), y] = 1.0
    return out

def load_mnist(seed: int = 42, val_size: int = 10_000):
    from tensorflow.keras.datasets import mnist
    (x_train, y_train), (x_test, y_test) = mnist.load_data()

    x_train = (x_train.astype(np.float32) / 255.0).reshape(-1, 784)
    x_test  = (x_test.astype(np.float32) / 255.0).reshape(-1, 784)

    rng = np.random.default_rng(seed)
    idx = np.arange(x_train.shape[0])
    rng.shuffle(idx)
    x_train, y_train = x_train[idx], y_train[idx]

    x_val, y_val = x_train[:val_size], y_train[:val_size]
    x_train, y_train = x_train[val_size:], y_train[val_size:]

    return (x_train, one_hot(y_train)), (x_val, one_hot(y_val)), (x_test, one_hot(y_test))


# PyTorch MLP

class TorchMLP(nn.Module):
    def __init__(self, in_dim: int, h1: int, h2: int, out_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.fc3 = nn.Linear(h2, out_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

def accuracy(model: nn.Module, x: torch.Tensor, y_onehot: torch.Tensor) -> float:
    model.eval()
    with torch.no_grad():
        logits = model(x)
        pred = torch.argmax(logits, dim=-1)
        true = torch.argmax(y_onehot, dim=-1)
        return float((pred == true).float().mean().item())

# Training
def run_experiment(x_train, y_train, x_val, y_val, x_test, y_test, batch_size: int):
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = TorchMLP(784, H1, H2, NUM_CLASSES).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    x_val_t = torch.tensor(x_val, dtype=torch.float32, device=device)
    y_val_t = torch.tensor(y_val, dtype=torch.float32, device=device)
    x_test_t = torch.tensor(x_test, dtype=torch.float32, device=device)
    y_test_t = torch.tensor(y_test, dtype=torch.float32, device=device)

    train_losses = []
    val_accs = []
    epoch_times = []

    n = x_train.shape[0]
    for ep in range(EPOCHS):
        t0 = time.perf_counter()
        model.train()

        rng = np.random.default_rng(SEED + ep)
        idx = np.arange(n)
        rng.shuffle(idx)

        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            bidx = idx[start:end]

            xb = torch.tensor(x_train[bidx], dtype=torch.float32, device=device)
            yb = torch.tensor(y_train[bidx], dtype=torch.float32, device=device)

            logits = model(xb)
            log_probs = F.log_softmax(logits, dim=-1)
            loss = -(yb * log_probs).sum(dim=-1).mean()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

        if device.type == "cuda":
            torch.cuda.synchronize()

        t1 = time.perf_counter()
        epoch_times.append(t1 - t0)

        # curves (small subset for train loss estimate)
        model.eval()
        with torch.no_grad():
            xb_small = torch.tensor(x_train[:2048], dtype=torch.float32, device=device)
            yb_small = torch.tensor(y_train[:2048], dtype=torch.float32, device=device)
            logits_small = model(xb_small)
            log_probs_small = F.log_softmax(logits_small, dim=-1)
            tr_loss = float((-(yb_small * log_probs_small).sum(dim=-1).mean()).item())

        va = accuracy(model, x_val_t, y_val_t)
        train_losses.append(tr_loss)
        val_accs.append(va)

    first_epoch = float(epoch_times[0])
    steady_state = float(np.mean(epoch_times[1:])) if EPOCHS > 1 else first_epoch
    test_acc = accuracy(model, x_test_t, y_test_t)

    return {
        "device": str(device),
        "first_epoch_time_s": first_epoch,
        "steady_state_epoch_time_s": steady_state,
        "final_test_acc": test_acc,
        "train_loss_curve": train_losses,
        "val_acc_curve": val_accs,
    }


# Main: run all batch sizes
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    (x_train, y_train), (x_val, y_val), (x_test, y_test) = load_mnist(SEED, VAL_SIZE)

    rows = []
    curves = {}

    for bs in BATCH_SIZES:
        res = run_experiment(x_train, y_train, x_val, y_val, x_test, y_test, bs)
        rows.append({
            "framework": "PyTorch",
            "batch_size": bs,
            "device": res["device"],
            "first_epoch_time_s": res["first_epoch_time_s"],
            "steady_state_epoch_time_s": res["steady_state_epoch_time_s"],
            "final_test_acc": res["final_test_acc"],
        })
        curves[str(bs)] = res

    df = pd.DataFrame(rows).sort_values(["batch_size"]).reset_index(drop=True)
    df.to_csv(os.path.join(OUT_DIR, "perf_results_torch.csv"), index=False)
    print("\n=== PyTorch Results ===")
    print(df)


    # Training Loss
    plt.figure()
    for bs in BATCH_SIZES:
        loss_curve = curves[str(bs)]["train_loss_curve"]
        plt.plot(loss_curve, label=f"bs={bs}")

    plt.xlabel("Epoch")
    plt.ylabel("Train Loss")
    plt.title("PyTorch Training Loss (All Batch Sizes)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "train_loss_all_batch_sizes.png"), dpi=150)
    plt.close()

    # Validation Accuracy
    plt.figure()
    for bs in BATCH_SIZES:
        acc_curve = curves[str(bs)]["val_acc_curve"]
        plt.plot(acc_curve, label=f"bs={bs}")

    plt.xlabel("Epoch")
    plt.ylabel("Validation Accuracy")
    plt.title("PyTorch Validation Accuracy (All Batch Sizes)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "val_acc_all_batch_sizes.png"), dpi=150)
    plt.close()

    print(f"\nSaved combined plots in: {OUT_DIR}/")

if __name__ == "__main__":
    main()


Writing assignment_torch.py


In [3]:
!python assignment_jax.py
!python assignment_torch.py


2026-02-14 04:58:02.914219: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771045082.936488     233 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771045082.942964     233 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771045082.958600     233 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771045082.958622     233 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771045082.958626     233 computation_placer.cc:177] computation placer alr

epoch with 50: 2026-02-14 03:42:52.768600: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
E0000 00:00:1771040572.789058   20639 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771040572.795781   20639 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771040572.811393   20639 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771040572.811417   20639 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771040572.811421   20639 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771040572.811424   20639 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.

=== JAX Results ===
  framework  batch_size  ...  steady_state_epoch_time_s  final_test_acc
0       JAX          64  ...                   0.779961          0.9370
1       JAX         256  ...                   0.241276          0.9017
2       JAX        1024  ...                   0.119667          0.8191

[3 rows x 5 columns]

Saved combined plots in: outputs_jax/
2026-02-14 03:44:08.199905: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
E0000 00:00:1771040648.220753   21071 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771040648.227624   21071 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771040648.245120   21071 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771040648.245146   21071 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771040648.245149   21071 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771040648.245152   21071 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
2026-02-14 03:44:08.249708: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.

=== PyTorch Results ===
  framework  batch_size  ... steady_state_epoch_time_s  final_test_acc
0   PyTorch          64  ...                  1.357513          0.9812
1   PyTorch         256  ...                  0.384666          0.9773
2   PyTorch        1024  ...                  0.146708          0.9784

[3 rows x 6 columns]

Saved combined plots in: outputs_torch/